In [ ]:
import pandas as pd
import numpy as np
import wandb

api = wandb.Api()

ENTITY  = "X"
PROJECT = "X"

runs = list(api.runs(f"{ENTITY}/{PROJECT}"))
print(f"Loaded {len(runs)} runs")

Loaded 200 runs


In [44]:
filters = {
    "config.dataset": "bios",
    "config.strategy": {"$in": ["bo", "disagreement"]},
    "$or": [
        {"config.batch_size": {"$in": [256, 512]}},
        {"config.batch_size": {"$in": ["256", "512"]}},
    ],
    "$or": [
        {"config.epochs_opt": {"$in": [8, 10]}},
        {"config.epochs_opt": {"$in": ["8", "10"]}},
    ],
}
runs = api.runs(f"{ENTITY}/{PROJECT}", filters=filters)


In [45]:
def as_int(x):
    try:
        return int(x)
    except Exception:
        return None

runs_raw = api.runs(
    f"{ENTITY}/{PROJECT}",
    filters={"config.dataset": "bios", "config.strategy": {"$in": ["bo", "disagreement"]}},
)

runs = []
for r in runs_raw:
    bs = as_int(r.config.get("batch_size"))  # or "batch_size" if that's the key you want
    ep = as_int(r.config.get("epochs_opt"))
    if bs in (256, 512) and ep in (8, 10):
        runs.append(r)

print("Matched runs:", len(runs))


Matched runs: 32


In [46]:
import pandas as pd

OUTER_KEYS = {
    "T_size": "audit/outer/T_size",
    "err_active": "audit/outer/err_mid",
    "width_abs": "audit/outer/width",
    "iter": "audit/outer/iter",

    "delta_est": "audit/outer/delta_mid",

    # keep bounds in history (these are per-iter)
    "h_min": "audit/cerm/delta_min_true",
    "h_max": "audit/cerm/delta_max_true",
}

HISTORY_KEYS = ["_step"] + list(OUTER_KEYS.values())

ts_rows = []
run_rows = []  # per-run scalars

for run in runs:
    h = pd.DataFrame(run.scan_history(keys=HISTORY_KEYS))
    if h.empty:
        continue

    # rename to short names
    h = h.rename(columns={v: k for k, v in OUTER_KEYS.items()})

    # add identifiers
    h["run_id"] = run.id
    h["seed"] = run.config.get("seed")
    h["strategy"] = run.config.get("strategy")
    h["surrogate_batch_size"] = run.config.get("surrogate_batch_size")
    h["epochs_opt"] = run.config.get("epochs_opt")

    # numeric coercion (include h_min/h_max)
    for c in ["T_size", "err_active", "width_abs", "iter", "h_min", "h_max", "_step"]:
        if c in h.columns:
            h[c] = pd.to_numeric(h[c], errors="coerce")

    # keep only valid iteration rows
    h = h.dropna(subset=["iter"])

    ts_rows.append(h)

    # --- per-run scalars from summary (broadcast later) ---
    run_rows.append({
        "run_id": run.id,
        "delta_bb_final": run.summary.get("audit/final/delta_bb"),
        "delta_bb_init":  run.summary.get("audit/init/delta_bb"),
    })

ts = pd.concat(ts_rows, ignore_index=True)

run_df = pd.DataFrame(run_rows)
ts = ts.merge(run_df, on="run_id", how="left")


In [47]:
ts.to_csv("bafa_trajectories_bios.csv", index=False)

In [48]:
import numpy as np
import pandas as pd

def _normalized_auec(
    df, t_col, err_col, t_max,
    t_start=0.0,
    prepend_t0=True,
):
    """
    Normalized AUEC over [t_start, t_max]:
      (1 / (t_max - t_start)) * ∫_{t_start}^{t_max} error(t) dt
    """
    d = df[[t_col, err_col]].dropna().copy()
    if d.empty:
        return np.nan

    d = d.sort_values(t_col).drop_duplicates(subset=[t_col], keep="last")

    # keep only within [t_start, t_max]
    d = d[(d[t_col] >= t_start) & (d[t_col] <= t_max)]
    if d.empty:
        return np.nan

    # prepend point at t_start (flat at first observed error in-window)
    if prepend_t0 and float(d[t_col].iloc[0]) > float(t_start):
        first_err = float(d[err_col].iloc[0])
        d = pd.concat(
            [pd.DataFrame({t_col: [t_start], err_col: [first_err]}), d],
            ignore_index=True
        )

    # extend to t_max (flat at last observed error in-window)
    if float(d[t_col].iloc[-1]) < float(t_max):
        last_err = float(d[err_col].iloc[-1])
        d = pd.concat(
            [d, pd.DataFrame({t_col: [t_max], err_col: [last_err]})],
            ignore_index=True
        )

    t = d[t_col].to_numpy(dtype=float)
    e = d[err_col].to_numpy(dtype=float)
    denom = float(t_max - t_start)
    if denom <= 0:
        return np.nan
    return float(np.trapz(e, t) / denom)


def _interp_err_at_budget(df, t_col, err_col, budget):
    d = df[[t_col, err_col]].dropna().copy().sort_values(t_col)
    if d.empty:
        return np.nan

    if budget in d[t_col].values:
        return float(d.loc[d[t_col] == budget, err_col].iloc[-1])

    before = d[d[t_col] <= budget]
    after  = d[d[t_col] >= budget]
    if len(before) == 0 or len(after) == 0:
        return np.nan

    t1, e1 = float(before[t_col].iloc[-1]), float(before[err_col].iloc[-1])
    t2, e2 = float(after[t_col].iloc[0]),  float(after[err_col].iloc[0])
    if t1 == t2:
        return float(e1)
    return float(e1 + (e2 - e1) * (budget - t1) / (t2 - t1))


def summarize_by_strategy(
    ts: pd.DataFrame,
    eps_list=(0.02, 0.05),
    t_max=2000,
    budget=250,
    t_col="T_size",
    err_col="err_active",
    seed_col="seed",
    strategy_col="strategy",
    t_start=42,  # <-- NEW: burn-in (e.g., 20 to ignore initial 4-query point)
) -> pd.DataFrame:
    """
    Same as before, but excludes all points with T < t_start from:
      - mean_crossing_T (population mean curve crossing)
      - per-replicate first-hit queries_to_eps
      - AUEC (computed over [t_start, t_max])
    """

    df = ts.copy()

    # numeric coercion
    for c in [t_col, err_col, seed_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[t_col, err_col, seed_col, strategy_col])

    if "run_id" not in df.columns:
        df["run_id"] = "NA"

    # Deduplicate per timepoint within each run
    df = df.sort_values([strategy_col, seed_col, "run_id", t_col])
    df = df.drop_duplicates(subset=[strategy_col, seed_col, "run_id", t_col], keep="last")

    out_rows = []

    for strategy, sdf_all in df.groupby(strategy_col, dropna=False):
        replicates = sdf_all[[seed_col, "run_id"]].drop_duplicates().values.tolist()

        auec_vals = []
        errB_vals = []
        first_hit = {eps: [] for eps in eps_list}

        for seed, run_id in replicates:
            rdf = sdf_all[(sdf_all[seed_col] == seed) & (sdf_all["run_id"] == run_id)].sort_values(t_col)

            # AUEC over [t_start, t_max]
            auec_vals.append(
                _normalized_auec(rdf, t_col=t_col, err_col=err_col, t_max=t_max, t_start=t_start)
            )

            # error at fixed budget (unchanged)
            errB_vals.append(_interp_err_at_budget(rdf, t_col=t_col, err_col=err_col, budget=budget))

            # first-hit AFTER burn-in
            rdf_eval = rdf[rdf[t_col] >= t_start]
            for eps in eps_list:
                hit = rdf_eval[rdf_eval[err_col] <= eps]
                first_hit[eps].append(float(hit[t_col].iloc[0]) if len(hit) else np.nan)

        # Population mean curve AFTER burn-in
        sdf_eval = sdf_all[(sdf_all[t_col] >= t_start) & (sdf_all[t_col] <= t_max)]
        pop_curve = (
            sdf_eval.groupby(t_col)[err_col]
                   .mean()
                   .reset_index()
                   .sort_values(t_col)
        )

        base = {
            "strategy": strategy,
            "n_replicates": int(len(replicates)),
            "auec_mean": float(np.nanmean(auec_vals)) if np.any(~np.isnan(auec_vals)) else np.nan,
            "auec_std":  float(np.nanstd(auec_vals))  if np.any(~np.isnan(auec_vals)) else np.nan,
            f"error_at_{budget}_mean": float(np.nanmean(errB_vals)) if np.any(~np.isnan(errB_vals)) else np.nan,
            f"error_at_{budget}_std":  float(np.nanstd(errB_vals))  if np.any(~np.isnan(errB_vals)) else np.nan,
        }

        for eps in eps_list:
            vals = np.array(first_hit[eps], dtype=float)
            valid = vals[~np.isnan(vals)]

            crossed = pop_curve[pop_curve[err_col] <= eps]
            mean_crossing_T = float(crossed[t_col].min()) if len(crossed) else np.nan

            out_rows.append({
                **base,
                "epsilon": float(eps),
                "mean_crossing_T": mean_crossing_T,
                f"queries_to_{eps}_mean": float(valid.mean()) if len(valid) else np.nan,
                f"queries_to_{eps}_std":  float(valid.std())  if len(valid) else np.nan,
                f"queries_to_{eps}_n_reached": int(len(valid)),
            })

    return pd.DataFrame(out_rows).sort_values(["strategy", "epsilon"]).reset_index(drop=True)


In [49]:
# If you only want bo + disagreement:
ts_f = ts[ts["strategy"].isin(["bo", "disagreement"])].copy()

summary = summarize_by_strategy(
    ts_f,
    eps_list=(0.02, 0.05),
    t_max=2000,
    budget=250,
    t_col="T_size",
    err_col="err_active",
)

print(summary)


       strategy  n_replicates  auec_mean  auec_std  error_at_250_mean  \
0            bo            15   0.025651  0.007874           0.019757   
1            bo            15   0.025651  0.007874           0.019757   
2  disagreement            17   0.024884  0.009231           0.021848   
3  disagreement            17   0.024884  0.009231           0.021848   

   error_at_250_std  epsilon  mean_crossing_T  queries_to_0.02_mean  \
0          0.007353     0.02            244.0                 128.8   
1          0.007353     0.05            180.0                   NaN   
2          0.008871     0.02            340.0                 130.0   
3          0.008871     0.05            148.0                   NaN   

   queries_to_0.02_std  queries_to_0.02_n_reached  queries_to_0.05_mean  \
0            73.483967                       15.0                   NaN   
1                  NaN                        NaN             94.666667   
2            61.676576                       16.0    